# 4 · Live predictions (15 minutes ahead)
Adapted from the professor's `5-predict-stations-FE`: builds features from the last 45 minutes of `live_station_status` with his `build_predictors()`, scores each station's newest poll with our `champion` models, and appends to `team1_live_predictions_15` for the map app.

**Schedule:** a job running `free_edition_models/0-live-station-poller-FE` then this notebook (`run_mode=production`). See the README.

In [ ]:
# Settings and shared code
from datetime import datetime, timezone

import mlflow
import mlflow.sklearn
import pandas as pd
from pyspark.sql import functions as F

import mvp_config as cfg
import mvp_lib as lib

lib.use_station_features()
from station_features import build_predictors

spark.conf.set("spark.sql.session.timeZone", "UTC")
dbutils.widgets.dropdown("run_mode", "dev", ["dev", "production"])
PUBLISH = dbutils.widgets.get("run_mode") == "production"
mlflow.set_registry_uri("databricks-uc")

info = spark.table(cfg.MODEL_INFO).toPandas().set_index("target")
models = {t: mlflow.sklearn.load_model(f"models:/{cfg.MODEL_NAMES[t]}@{cfg.MODEL_ALIAS}") for t in ("bike", "dock")}
thresholds = {t: float(info.loc[t, "threshold"]) for t in ("bike", "dock")}
versions = {t: str(info.loc[t, "version"]) for t in ("bike", "dock")}

In [ ]:
# Features for the newest poll of each station (same steps as the professor's notebook).
window_end = F.current_timestamp()
window_start = window_end - F.expr(f"INTERVAL {cfg.LIVE_HISTORY_MIN + cfg.MAX_STALENESS_MIN} MINUTES")
recent_status = spark.table(cfg.LIVE_STATUS).filter(F.col("fetched_at") >= window_start)

columns = sorted(set(lib.all_input_columns()) | {"station_id", "fetched_at", "station_name", "capacity",
                                                  "bike_stockout", "dock_stockout",
                                                  "num_bikes_available", "num_docks_available"})
recent = (build_predictors(recent_status, spark.table(cfg.LIVE_INFO), local_timezone=cfg.LOCAL_TZ)
          .select(*columns).toPandas())
if recent.empty:
    raise ValueError("No live polls in the last hour. Run free_edition_models/0-live-station-poller-FE first.")

prediction_time = recent.fetched_at.max()
newest = recent.sort_values("fetched_at").groupby("station_id").tail(1)
newest = newest[newest.fetched_at >= prediction_time - pd.Timedelta(minutes=cfg.MAX_STALENESS_MIN)]
print(f"newest poll {prediction_time} UTC · {len(newest):,} stations with a fresh poll")

In [ ]:
# Score and save once per poll time.
scored = lib.score_stations(newest.reset_index(drop=True), models, thresholds)
out = scored[["station_id", "fetched_at", "num_bikes_available", "num_docks_available", "capacity",
              "bike_state", "dock_state", "p_bike", "p_dock", "status"]].copy()
out["prediction_time"] = prediction_time
out["predicted_at"] = datetime.now(timezone.utc).replace(tzinfo=None)
out["bike_model_version"], out["dock_model_version"] = versions["bike"], versions["dock"]
print(out["status"].value_counts().to_string())

if PUBLISH:
    new = spark.createDataFrame(out)
    new = new.withColumns({c: F.when(~F.isnan(c), F.col(c)) for c in ("p_bike", "p_dock")})
    exists = spark.catalog.tableExists(cfg.LIVE_PREDICTIONS)
    if exists and not spark.table(cfg.LIVE_PREDICTIONS).filter(
            F.col("prediction_time") == F.lit(prediction_time.to_pydatetime())).isEmpty():
        print("already scored this poll; nothing written")
    else:
        new.write.mode("append").saveAsTable(cfg.LIVE_PREDICTIONS)
        print(f"{len(out):,} rows appended to {cfg.LIVE_PREDICTIONS}")
    spark.sql(f"DELETE FROM {cfg.LIVE_PREDICTIONS} WHERE prediction_time < current_timestamp() - INTERVAL 14 DAYS")
else:
    print("dev: nothing written")

display(out.sort_values("p_bike", ascending=False).head(15))